## 6. Mô hình hồi quy dự đoán freight_value

Mục này xây dựng mô hình Random Forest với 4 biến đầu vào, so sánh với mô hình nền Linear Regression và đánh giá bằng MAE, RMSE, R².

### 6.1 Đọc dữ liệu

Cell dưới gắn Google Drive và đọc file olist_clean.csv trong thư mục output.

In [ ]:
import glob
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

drive.mount('/content/drive')
duong_dan = glob.glob('/content/drive/MyDrive/**/output/olist_clean.csv', recursive=True)
df = pd.read_csv(duong_dan[0])
print(df.shape)

Mounted at /content/drive
(108937, 10)


### 6.2 Biến mục tiêu và biến đầu vào

**Biến mục tiêu:** freight_value (BRL).

**Biến đầu vào:**
- weight_g: cân nặng sản phẩm.
- volume_cm3: thể tích sản phẩm.
- distance_km: khoảng cách giữa người bán và người mua.
- price: giá bán.

**Lý do chọn:** cân nặng, thể tích và khoảng cách là các yếu tố vật lý quyết định phí vận chuyển. Giá bán được thêm vào để kiểm tra giá có đóng góp thông tin hay không.

Cell dưới tách bảng thành X (đầu vào) và y (mục tiêu) và kiểm tra giá trị thiếu.

In [ ]:
cot_X = ['weight_g', 'volume_cm3', 'distance_km', 'price']
X = df[cot_X]
y = df['freight_value']
print(X.shape, y.shape)
print(X.isna().sum())

(108937, 4) (108937,)
weight_g       0
volume_cm3     0
distance_km    0
price          0
dtype: int64


### 6.3 Chia tập train và test

**Cách làm:** 80% dữ liệu để huấn luyện, 20% để kiểm tra, cố định random_state = 42 để kết quả lặp lại được.

**Lưu ý:** việc chia dữ liệu được thực hiện trước khi huấn luyện để tập test không tham gia vào quá trình học của mô hình.

Cell dưới chia dữ liệu và in kích thước hai tập.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(87149, 4) (21788, 4)


### 6.4 Mô hình nền: Linear Regression

**Mục đích:** tạo mốc so sánh. Nếu Random Forest không vượt mô hình tuyến tính thì không cần dùng mô hình phức tạp hơn.

Cell dưới huấn luyện Linear Regression và tính MAE, RMSE, R² trên tập test.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print('MAE  :', round(mean_absolute_error(y_test, pred_lr), 2), 'BRL')
print('RMSE :', round(mean_squared_error(y_test, pred_lr) ** 0.5, 2), 'BRL')
print('R2   :', round(r2_score(y_test, pred_lr), 3))

MAE  : 5.22 BRL
RMSE : 9.97 BRL
R2   : 0.59


### 6.5 Mô hình Random Forest

**Cách làm:** rừng gồm 100 cây, mỗi cây học trên một mẫu ngẫu nhiên của tập train, kết quả dự đoán là trung bình của 100 cây.

**Lý do chọn:** quan hệ giữa phí vận chuyển với cân nặng, thể tích và khoảng cách không tuyến tính. Random Forest không yêu cầu chuẩn hoá dữ liệu.

Cell dưới huấn luyện Random Forest và tính MAE, RMSE, R² trên tập test.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print('MAE  :', round(mean_absolute_error(y_test, pred_rf), 2), 'BRL')
print('RMSE :', round(mean_squared_error(y_test, pred_rf) ** 0.5, 2), 'BRL')
print('R2   :', round(r2_score(y_test, pred_rf), 3))

MAE  : 3.6 BRL
RMSE : 7.76 BRL
R2   : 0.751


### 6.6 Độ quan trọng của biến

**Mục đích:** xác định biến nào đóng góp nhiều nhất vào dự đoán của Random Forest, làm cơ sở trả lời RQ1, RQ2 và RQ4.

Cell dưới in độ quan trọng của 4 biến, sắp xếp giảm dần. Tổng các giá trị bằng 1.

In [ ]:
do_quan_trong = pd.Series(rf.feature_importances_, index=cot_X)
print(do_quan_trong.sort_values(ascending=False).round(3))

volume_cm3     0.348
distance_km    0.310
weight_g       0.214
price          0.129
dtype: float64


### 6.7 Kết luận

**Kết luận:** Random Forest có sai số thấp hơn và R² cao hơn Linear Regression ở cả ba chỉ số trên tập test.

| Mô hình | MAE (BRL) | RMSE (BRL) | R² |
|---|---|---|---|
| Linear Regression | 5.22 | 9.97 | 0.590 |
| Random Forest | 3.60 | 7.76 | 0.751 |

| Biến | Độ quan trọng |
|---|---|
| volume_cm3 | 0.348 |
| distance_km | 0.310 |
| weight_g | 0.214 |
| price | 0.129 |